# Comparison Report Demo

- **Muc tieu:** Tuan 7-8 — sinh bao cao so sanh + citation
- **Nguyen tac:** _Khong bang chung -> khong ket luan_
- **Du lieu:** 2 file DOCX/PDF (sample v1 va v2)

Pipeline: **DOCX/PDF** → Read → Normalize → Chunk → Embedding → So sanh LLM → Bao cao

Notebook nay thuc hien full pipeline tu file goc (DOCX/PDF) den bao cao so sanh co trich dan.


In [1]:
# === Cell 2: Imports + Config ===
import sys
from pathlib import Path
import json
import re
import os
import difflib
from collections import Counter

import pandas as pd
from IPython.display import HTML, display
from dotenv import load_dotenv

try:
    import ollama
except ImportError:
    ollama = None

_HTML = HTML


def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'sample_pairs'
PAIRS_DIR = DATA_DIR / 'pairs'
OUTPUT_DIR = ROOT / 'outputs'
CHROMA_DIR = ROOT / 'chroma_db'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# --- Dau vao: 2 file DOCX hoac PDF ---
FILE_V1 = PAIRS_DIR / 'sample v1.docx'
FILE_V2 = PAIRS_DIR / 'sample v2.docx'

MANUAL_CHANGES_MD = PAIRS_DIR / 'sample_v2_changes.md'
REPORT_JSON = OUTPUT_DIR / 'sample_pair_comparison_report.json'

LLM_MODEL = 'qwen2.5:7b-instruct-q4_K_M'
MAX_COMPARE_UNITS = None

load_dotenv(ROOT / '.env')
EMBED_MODEL = 'huyydangg/DEk21_hcmute_embedding'

assert FILE_V1.exists(), f'Khong tim thay {FILE_V1}'
assert FILE_V2.exists(), f'Khong tim thay {FILE_V2}'
assert MANUAL_CHANGES_MD.exists(), f'Khong tim thay {MANUAL_CHANGES_MD}'

print(f'[OK] file v1          : {FILE_V1.name}')
print(f'[OK] file v2          : {FILE_V2.name}')
print(f'[OK] manual changes   : {MANUAL_CHANGES_MD}')
print(f'[OK] report json      : {REPORT_JSON}')
print(f'[CFG] llm model       : {LLM_MODEL}')
print(f'[CFG] embed model     : {EMBED_MODEL}')
print(f'[CFG] max compare unit: {MAX_COMPARE_UNITS}')


[OK] file v1          : sample v1.docx
[OK] file v2          : sample v2.docx
[OK] manual changes   : d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\data\sample_pairs\pairs\sample_v2_changes.md
[OK] report json      : d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\outputs\sample_pair_comparison_report.json
[CFG] llm model       : qwen2.5:7b-instruct-q4_K_M
[CFG] embed model     : huyydangg/DEk21_hcmute_embedding
[CFG] max compare unit: None


In [2]:
# === Cell 3: Doc DOCX/PDF -> Normalize -> Chunk theo Dieu ===
# (Cac ham duoc lay tu Ingestion_pipeline_test.ipynb)

import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime, timezone

from docx import Document
import pdfplumber

# ── Doc DOCX/PDF ──────────────────────────────────────────────────────

W_URI = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
W_NS = {"w": W_URI}


def _w_attr(name: str) -> str:
    return f"{{{W_URI}}}{name}"


def _read_docx_xml(path: Path, inner_path: str):
    with zipfile.ZipFile(path) as zf:
        try:
            data = zf.read(inner_path)
        except KeyError:
            return None
    return ET.fromstring(data)


def _parse_docx_style_map(path: Path) -> dict:
    tree = _read_docx_xml(path, "word/styles.xml")
    if tree is None:
        return {}
    styles = {}
    for style in tree.findall(".//w:style", W_NS):
        style_id = style.get(_w_attr("styleId"))
        if not style_id:
            continue
        name_el = style.find("w:name", W_NS)
        based_on_el = style.find("w:basedOn", W_NS)
        num_pr = style.find("w:pPr/w:numPr", W_NS)
        num_id, ilvl = None, None
        if num_pr is not None:
            nid_el = num_pr.find("w:numId", W_NS)
            il_el = num_pr.find("w:ilvl", W_NS)
            if nid_el is not None:
                num_id = nid_el.get(_w_attr("val"))
            if il_el is not None:
                ilvl = il_el.get(_w_attr("val"))
        styles[style_id] = {
            "style_id": style_id,
            "style_name": name_el.get(_w_attr("val")) if name_el is not None else style_id,
            "based_on": based_on_el.get(_w_attr("val")) if based_on_el is not None else None,
            "num_id": int(num_id) if num_id is not None else None,
            "ilvl": int(ilvl) if ilvl is not None else None,
        }
    return styles


def _resolve_style_numbering(style_id, style_map, seen=None):
    if not style_id or style_id not in style_map:
        return None, None
    seen = seen or set()
    if style_id in seen:
        return None, None
    seen.add(style_id)
    info = style_map[style_id]
    if info["num_id"] is not None:
        return info["num_id"], info["ilvl"] if info["ilvl"] is not None else 0
    return _resolve_style_numbering(info["based_on"], style_map, seen)


def _parse_docx_numbering(path: Path) -> dict:
    tree = _read_docx_xml(path, "word/numbering.xml")
    numbering = {"num_to_abstract": {}, "abstract_levels": {}}
    if tree is None:
        return numbering
    for abstract in tree.findall("w:abstractNum", W_NS):
        abstract_id = abstract.get(_w_attr("abstractNumId"))
        if abstract_id is None:
            continue
        levels = {}
        for lvl in abstract.findall("w:lvl", W_NS):
            ilvl_raw = lvl.get(_w_attr("ilvl"))
            if ilvl_raw is None:
                continue
            start_el = lvl.find("w:start", W_NS)
            fmt_el = lvl.find("w:numFmt", W_NS)
            text_el = lvl.find("w:lvlText", W_NS)
            levels[int(ilvl_raw)] = {
                "start": int(start_el.get(_w_attr("val"))) if start_el is not None else 1,
                "num_fmt": fmt_el.get(_w_attr("val")) if fmt_el is not None else "decimal",
                "lvl_text": text_el.get(_w_attr("val")) if text_el is not None else f"%{int(ilvl_raw) + 1}",
            }
        numbering["abstract_levels"][int(abstract_id)] = levels
    for num in tree.findall("w:num", W_NS):
        num_id = num.get(_w_attr("numId"))
        abstract_el = num.find("w:abstractNumId", W_NS)
        if num_id is None or abstract_el is None:
            continue
        numbering["num_to_abstract"][int(num_id)] = int(abstract_el.get(_w_attr("val")))
    return numbering


def _to_roman(value: int) -> str:
    pairs = [(1000,"M"),(900,"CM"),(500,"D"),(400,"CD"),(100,"C"),(90,"XC"),
             (50,"L"),(40,"XL"),(10,"X"),(9,"IX"),(5,"V"),(4,"IV"),(1,"I")]
    result, remaining = [], max(1, value)
    for arabic, roman in pairs:
        while remaining >= arabic:
            result.append(roman)
            remaining -= arabic
    return "".join(result)


def _to_alpha(value: int, uppercase=False) -> str:
    chars, remaining = [], max(1, value)
    while remaining > 0:
        remaining -= 1
        chars.append(chr(ord("A" if uppercase else "a") + (remaining % 26)))
        remaining //= 26
    return "".join(reversed(chars))


def _format_counter(value: int, num_fmt: str) -> str:
    if num_fmt in {"upperRoman", "romanUpper"}:
        return _to_roman(value)
    if num_fmt in {"lowerRoman", "romanLower"}:
        return _to_roman(value).lower()
    if num_fmt == "upperLetter":
        return _to_alpha(value, True)
    if num_fmt == "lowerLetter":
        return _to_alpha(value, False)
    return str(value)


def _render_numbering_label(num_id, ilvl, numbering_data, counters_by_num):
    abstract_id = numbering_data["num_to_abstract"].get(num_id)
    levels = numbering_data["abstract_levels"].get(abstract_id, {}) if abstract_id is not None else {}
    level_info = levels.get(ilvl, {})
    counters = counters_by_num[num_id]
    start = level_info.get("start", 1)
    counters[ilvl] = counters.get(ilvl, start - 1) + 1
    for level in list(counters):
        if level > ilvl:
            del counters[level]
    label = level_info.get("lvl_text", f"%{ilvl + 1}")
    path_parts = []
    for level in range(ilvl + 1):
        if level not in counters:
            continue
        num_fmt = levels.get(level, {}).get("num_fmt", "decimal")
        formatted = _format_counter(counters[level], num_fmt)
        label = label.replace(f"%{level + 1}", formatted)
        path_parts.append(formatted)
    return re.sub(r"\s+", " ", label).strip(), path_parts


def _extract_paragraph_numbering(paragraph):
    p_pr = paragraph._p.pPr
    num_pr = p_pr.numPr if p_pr is not None else None
    if num_pr is None:
        return None, None
    num_id_el, ilvl_el = num_pr.numId, num_pr.ilvl
    return (int(num_id_el.val) if num_id_el is not None else None,
            int(ilvl_el.val) if ilvl_el is not None else 0)


def _get_heading_level(style_name, style_id):
    for candidate in (style_name, style_id):
        if not candidate:
            continue
        normalized = re.sub(r"\s+", "", candidate).lower()
        if normalized.startswith("heading") and normalized[7:].isdigit():
            return int(normalized[7:])
    return None


def read_docx_document(path: Path) -> dict:
    doc = Document(path)
    style_map = _parse_docx_style_map(path)
    numbering_data = _parse_docx_numbering(path)
    counters_by_num = defaultdict(dict)
    paragraphs = []
    for idx, paragraph in enumerate(doc.paragraphs):
        text = paragraph.text.strip()
        if not text:
            continue
        style_name = paragraph.style.name if paragraph.style else None
        style_id = paragraph.style.style_id if paragraph.style else None
        heading_level = _get_heading_level(style_name, style_id)
        num_id, ilvl = _extract_paragraph_numbering(paragraph)
        if num_id is None:
            num_id, ilvl = _resolve_style_numbering(style_id, style_map)
        numbering_label, numbering_path = None, []
        if num_id is not None and ilvl is not None:
            numbering_label, numbering_path = _render_numbering_label(
                num_id, ilvl, numbering_data, counters_by_num)
        display_text = text
        if heading_level in {1, 2, 3} and numbering_label:
            display_text = f"{numbering_label} {text}".strip()
        paragraphs.append({
            "idx": idx, "text": text, "display_text": display_text,
            "style_name": style_name, "style_id": style_id,
            "heading_level": heading_level, "num_id": num_id, "ilvl": ilvl,
            "numbering_label": numbering_label, "numbering_path": numbering_path,
        })
    return {"kind": "docx", "text": "\n".join(p["display_text"] for p in paragraphs), "paragraphs": paragraphs}


def read_pdf_text(path: Path) -> str:
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text.strip())
    return "\n".join(pages)


def read_document(path: Path) -> dict:
    ext = path.suffix.lower()
    if ext == ".docx":
        return read_docx_document(path)
    if ext == ".pdf":
        text = read_pdf_text(path)
        paragraphs = [
            {"idx": i, "text": l.strip(), "display_text": l.strip(),
             "style_name": None, "style_id": None, "heading_level": None,
             "num_id": None, "ilvl": None, "numbering_label": None, "numbering_path": []}
            for i, l in enumerate(text.splitlines()) if l.strip()
        ]
        return {"kind": "pdf", "text": text, "paragraphs": paragraphs}
    raise ValueError(f"Khong ho tro dinh dang: {ext}")


# ── Normalize ─────────────────────────────────────────────────────────

def normalize_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_document(document):
    if isinstance(document, str):
        return normalize_text(document)
    paragraphs = []
    for para in document.get("paragraphs", []):
        text = normalize_text(para.get("text", ""))
        display_text = normalize_text(para.get("display_text") or text)
        if not display_text:
            continue
        normalized = dict(para)
        normalized["text"] = text
        normalized["display_text"] = display_text
        paragraphs.append(normalized)
    return {**document, "paragraphs": paragraphs,
            "text": "\n".join(p["display_text"] for p in paragraphs)}


# ── Chunking theo Dieu ────────────────────────────────────────────────

CHUONG_RE = re.compile(r"^Chương\s+([IVXLCDM\d]+)[\.:]?\s*(.*)", re.IGNORECASE)
MUC_RE = re.compile(r"^Mục\s+(\d+)[\.:]?\s*(.*)", re.IGNORECASE)
DIEU_RE = re.compile(r"^(?:Điều|ĐIỀU|dieu|DIEU)\s+((?:\d+[A-Za-z]?|[IVXLCDM]+))[\.:]?\s*(.*)$")
KHOAN_RE = re.compile(r"^(\d+)[\.)]\s*(.*)$")
DIEM_RE = re.compile(r"^([a-zđ])[\.)]\s*(.*)$", re.IGNORECASE)
DINH_NGHIA_RE = re.compile(r"giải thích từ ngữ|định nghĩa|diễn giải", re.IGNORECASE)


def _clean_heading(line):
    return re.sub(r"[:\s]+$", "", line.strip())


def _looks_upper_heading(line):
    s = line.strip()
    if not s or s.endswith((".", ";", ",", ":")):
        return False
    if any(ch.isdigit() for ch in s):
        return False
    words = s.split()
    if len(words) < 2 or len(words) > 14:
        return False
    letters = [ch for ch in s if ch.isalpha()]
    return bool(letters) and sum(ch.isupper() for ch in letters) / len(letters) >= 0.8


def _looks_title_heading(line):
    s = line.strip()
    if not s or s.endswith((".", ";", ",", ":")):
        return False
    if any(ch.isdigit() for ch in s):
        return False
    if any(ch in s for ch in ('"', "'", "(", ")", "-", "•", "*")):
        return False
    words = [re.sub(r"[^\wÀ-ỹĐđ-]", "", w) for w in s.split()]
    words = [w for w in words if w]
    if len(words) < 2 or len(words) > 14:
        return False
    return sum(1 for w in words if w[0].isupper()) >= max(2, len(words) - 1)


def _is_contract_section_heading(line):
    s = line.strip()
    if not s:
        return False
    if CHUONG_RE.match(s) or MUC_RE.match(s) or DIEU_RE.match(s):
        return False
    if KHOAN_RE.match(s) or DIEM_RE.match(s):
        return False
    if s.startswith(("(", "-", "•", "*", '"', "\u201c")):
        return False
    if len(s) > 120:
        return False
    return _looks_upper_heading(s) or _looks_title_heading(s)


def _extract_khoan_items(body_lines):
    items, current = [], None
    for raw in body_lines:
        line = raw.strip()
        if not line:
            if current and current["text"]:
                current["text"] += "\n"
            continue
        km = KHOAN_RE.match(line)
        if km:
            if current:
                current["text"] = current["text"].strip()
                items.append(current)
            current = {"khoan_number": km.group(1), "text": km.group(2).strip(),
                       "diem_items": [], "tieu_muc_items": []}
            continue
        dm = DIEM_RE.match(line)
        if dm and current is not None:
            diem_val = dm.group(1).strip()
            current["diem_items"].append({"diem_number": diem_val, "diem_key": diem_val, "text": dm.group(2).strip()})
            current["text"] = (current["text"] + "\n" + line).strip()
            continue
        if current is not None:
            current["text"] = (current["text"] + "\n" + line).strip()
    if current:
        current["text"] = current["text"].strip()
        items.append(current)
    return items


def _build_chunk(*, chunk_idx, doc_id, version, chuong_so, muc_so,
                 clause_id, article_number, article_title, text,
                 khoan_items=None, is_header=False):
    khoan_items = khoan_items or []
    diem_count = sum(len(i.get("diem_items", [])) for i in khoan_items)
    tieu_muc_count = sum(len(i.get("tieu_muc_items", [])) for i in khoan_items)
    return {
        "chunk_id": f"{doc_id}_{version}_{chunk_idx:04d}",
        "doc_id": doc_id, "version": version,
        "chunk_type": "definition" if DINH_NGHIA_RE.search(article_title) else "article",
        "chuong_so": chuong_so, "muc_so": muc_so, "clause_id": clause_id,
        "article_number": article_number, "article_title": article_title,
        "khoan_count": 0 if is_header else len(khoan_items),
        "diem_count": 0 if is_header else diem_count,
        "tieu_muc_count": 0 if is_header else tieu_muc_count,
        "khoan_items": [] if is_header else khoan_items,
        "text": text, "char_len": len(text),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }


def _chunk_docx_headings(document, doc_id, version):
    paragraphs = document.get("paragraphs", [])
    chunks, chunk_idx = [], 1
    article_state, current_khoan, current_tieu_muc = None, None, None

    def flush():
        nonlocal article_state, current_khoan, current_tieu_muc, chunk_idx
        if article_state is None:
            return
        block = "\n".join(
            l for l in [article_state["heading_display"]] + article_state["body_lines"] if l.strip()
        ).strip()
        if block:
            chunks.append(_build_chunk(
                chunk_idx=chunk_idx, doc_id=doc_id, version=version,
                chuong_so="0", muc_so="0",
                clause_id=f"article_{article_state['article_number']}",
                article_number=article_state["article_number"],
                article_title=article_state["article_title"],
                text=block, khoan_items=article_state["khoan_items"],
            ))
            chunk_idx += 1
        article_state = current_khoan = current_tieu_muc = None

    for para in paragraphs:
        text = para.get("text", "").strip()
        dt = para.get("display_text") or text
        hl = para.get("heading_level")
        nl = para.get("numbering_label")
        np_ = para.get("numbering_path") or []

        if hl == 1 and nl:
            flush()
            article_state = {
                "article_number": np_[0] if np_ else nl.rstrip("."),
                "article_title": _clean_heading(text),
                "heading_display": dt.strip(), "body_lines": [], "khoan_items": [],
            }
            continue
        if article_state is None:
            continue
        if hl == 2 and nl:
            current_khoan = {"khoan_number": nl.rstrip("."), "title": _clean_heading(text),
                             "text": dt.strip(), "diem_items": [], "tieu_muc_items": []}
            current_tieu_muc = None
            article_state["khoan_items"].append(current_khoan)
            article_state["body_lines"].append(dt.strip())
            continue
        if hl == 3 and nl:
            if current_khoan is None:
                syn = ".".join(np_[:2]) if len(np_) >= 2 else nl.rstrip(".")
                current_khoan = {"khoan_number": syn, "title": "", "text": "",
                                 "diem_items": [], "tieu_muc_items": []}
                article_state["khoan_items"].append(current_khoan)
            current_tieu_muc = {"tieu_muc_number": nl.rstrip("."),
                                "title": _clean_heading(text), "text": dt.strip()}
            current_khoan["tieu_muc_items"].append(current_tieu_muc)
            current_khoan["text"] = "\n".join(p for p in [current_khoan["text"], dt.strip()] if p).strip()
            article_state["body_lines"].append(dt.strip())
            continue
        article_state["body_lines"].append(dt.strip())
        if current_tieu_muc is not None:
            current_tieu_muc["text"] = (current_tieu_muc["text"] + "\n" + dt.strip()).strip()
            current_khoan["text"] = (current_khoan["text"] + "\n" + dt.strip()).strip()
        elif current_khoan is not None:
            current_khoan["text"] = (current_khoan["text"] + "\n" + dt.strip()).strip()

    flush()
    return chunks


def _chunk_plain_text(text, doc_id, version):
    lines = [l for l in text.split("\n") if l.strip()]
    chunks, chunk_idx, section_seq = [], 1, 1
    cur_chuong, cur_muc = "0", "0"
    use_explicit = sum(1 for l in lines if DIEU_RE.match(l.strip())) > 0
    cur_no = cur_cid = None
    cur_head, cur_title, cur_body = [], [], []
    preamble_done = False

    def start(heading, article_number=None, title_text=None):
        nonlocal cur_no, cur_cid, cur_head, cur_title, cur_body, section_seq
        if article_number is None:
            cur_no, cur_cid = str(section_seq), f"section_{section_seq:03d}"
            section_seq += 1
        else:
            cur_no, cur_cid = str(article_number), f"article_{article_number}"
        cur_head = [heading.strip()]
        cur_title = [_clean_heading(title_text or heading)] if (title_text or heading).strip() else []
        cur_body = []

    def flush():
        nonlocal chunk_idx, cur_no, cur_cid, cur_head, cur_title, cur_body
        if cur_no is None:
            return
        if cur_cid.startswith("section_") and not any(l.strip() for l in cur_body):
            cur_no = cur_cid = None; cur_head = cur_title = cur_body = []; return
        block = "\n".join(l for l in cur_head + cur_body if l.strip()).strip()
        if not block:
            cur_no = cur_cid = None; cur_head = cur_title = cur_body = []; return
        at = " - ".join(l for l in cur_title if l.strip()) or " - ".join(_clean_heading(l) for l in cur_head)
        chunks.append(_build_chunk(
            chunk_idx=chunk_idx, doc_id=doc_id, version=version,
            chuong_so=cur_chuong, muc_so=cur_muc, clause_id=cur_cid,
            article_number=cur_no, article_title=at,
            text=block, khoan_items=_extract_khoan_items(cur_body),
        ))
        chunk_idx += 1
        cur_no = cur_cid = None; cur_head = cur_title = cur_body = []

    for line in lines:
        s = line.strip()
        mc = CHUONG_RE.match(s)
        mm = MUC_RE.match(s)
        md = DIEU_RE.match(s)
        if mc:
            flush(); cur_chuong = mc.group(1); cur_muc = "0"; preamble_done = True; continue
        if mm:
            flush(); cur_muc = mm.group(1); preamble_done = True; continue
        if md:
            flush()
            start(s, article_number=md.group(1), title_text=md.group(2).strip() or f"Điều {md.group(1)}")
            preamble_done = True; continue
        if use_explicit:
            if cur_no is not None:
                cur_body.append(s)
            continue
        if _is_contract_section_heading(s) and preamble_done:
            if cur_no is None:
                start(s)
            elif cur_body:
                flush(); start(s)
            else:
                cur_head.append(s); cur_title.append(_clean_heading(s))
            continue
        if cur_no is not None:
            cur_body.append(s)

    flush()
    return chunks


def chunk_full_hierarchy(document, doc_id, version):
    if isinstance(document, dict):
        paragraphs = document.get("paragraphs", [])
        if sum(1 for p in paragraphs if p.get("heading_level") == 1 and p.get("numbering_label")) > 0:
            return _chunk_docx_headings(document, doc_id, version)
        return _chunk_plain_text(document.get("text", ""), doc_id, version)
    return _chunk_plain_text(document, doc_id, version)


# ── Thuc thi ──────────────────────────────────────────────────────────

raw_v1 = read_document(FILE_V1)
raw_v2 = read_document(FILE_V2)
print(f'[OK] Doc file v1: {raw_v1["kind"]}, {len(raw_v1["paragraphs"])} paragraphs')
print(f'[OK] Doc file v2: {raw_v2["kind"]}, {len(raw_v2["paragraphs"])} paragraphs')

clean_v1 = normalize_document(raw_v1)
clean_v2 = normalize_document(raw_v2)

chunks_v1 = chunk_full_hierarchy(clean_v1, doc_id=FILE_V1.stem, version='v1')
chunks_v2 = chunk_full_hierarchy(clean_v2, doc_id=FILE_V2.stem, version='v2')

n_article_v1 = sum(1 for c in chunks_v1 if c.get('chunk_type') == 'article')
n_article_v2 = sum(1 for c in chunks_v2 if c.get('chunk_type') == 'article')

print(f'\n[OK] Chunks v1: {len(chunks_v1)} (article={n_article_v1})')
print(f'[OK] Chunks v2: {len(chunks_v2)} (article={n_article_v2})')
print(f'Mau chunk v1: Dieu {chunks_v1[0]["article_number"]} - {chunks_v1[0]["article_title"][:60]}')


[OK] Doc file v1: docx, 184 paragraphs
[OK] Doc file v2: docx, 184 paragraphs

[OK] Chunks v1: 21 (article=21)
[OK] Chunks v2: 21 (article=21)
Mau chunk v1: Dieu 1 - Đối Tượng Của Hợp Đồng


In [3]:
# === Cell 4: Embedding chunks ===

from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer
import chromadb


def load_embedder(model_name, hf_token=None):
    return SentenceTransformer(model_name, token=hf_token)


def get_collection(chroma_dir, collection_name="legal_chunks"):
    client = chromadb.PersistentClient(path=str(chroma_dir))
    return client.get_or_create_collection(collection_name)


def embed_chunks(chunks, embedder):
    texts = [c["text"] for c in chunks]
    segmented = [ViTokenizer.tokenize(t) for t in texts]
    vectors = embedder.encode(segmented, convert_to_numpy=True, show_progress_bar=True)
    return [v.tolist() for v in vectors]


def index_chunks(chunks, chroma_dir, embedder,
                 collection_name="legal_chunks", batch_size=64):
    if not chunks:
        return 0
    collection = get_collection(chroma_dir, collection_name)
    vectors = embed_chunks(chunks, embedder)
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        batch = chunks[start:end]
        collection.upsert(
            ids=[c["chunk_id"] for c in batch],
            documents=[c["text"] for c in batch],
            metadatas=[{
                "doc_id": c["doc_id"], "version": c["version"],
                "chuong_so": c["chuong_so"], "muc_so": c["muc_so"],
                "article_number": c["article_number"], "article_title": c["article_title"],
                "khoan_count": c["khoan_count"], "diem_count": c["diem_count"],
                "tieu_muc_count": c.get("tieu_muc_count", 0),
            } for c in batch],
            embeddings=vectors[start:end],
        )
    return len(chunks)


# ── Thuc thi ──────────────────────────────────────────────────────────

hf_token = os.getenv('HF_TOKEN')
embedder = load_embedder(model_name=EMBED_MODEL, hf_token=hf_token)
print(f'[OK] Embedding model: {embedder.get_sentence_embedding_dimension()} dim')

# Embed ca 2 ban
vectors_v1 = embed_chunks(chunks_v1, embedder)
vectors_v2 = embed_chunks(chunks_v2, embedder)

print(f'[OK] Embedded v1: {len(vectors_v1)} vectors')
print(f'[OK] Embedded v2: {len(vectors_v2)} vectors')

# Index vao ChromaDB (de phuc vu retrieval sau nay)
n1 = index_chunks(chunks_v1, chroma_dir=CHROMA_DIR, embedder=embedder)
n2 = index_chunks(chunks_v2, chroma_dir=CHROMA_DIR, embedder=embedder)
print(f'[OK] Indexed ChromaDB: v1={n1}, v2={n2}')


d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\.venv\Lib\site-packages\pyvi\ViTokenizer.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  model = pickle.load(fin)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 723.93it/s, Materializing param=pooler.dense.weight]                               


[OK] Embedding model: 768 dim


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.25it/s]


[OK] Embedded v1: 21 vectors
[OK] Embedded v2: 21 vectors


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.29it/s]


[OK] Indexed ChromaDB: v1=21, v2=21


In [5]:
# === Cell 5: So sanh o muc DIEU bang LLM + VECTOR RETRIEVAL ===

VECTOR_TOP_K = 3
VECTOR_MATCH_THRESHOLD = 0.50


def article_sort_key(article_no: str):
    text = str(article_no)
    match = re.match(r'^(\d+)([A-Za-z]*)$', text)
    if match:
        return (0, int(match.group(1)), match.group(2))
    return (1, text)


def build_articles_from_chunks(chunks: list[dict]) -> dict[str, dict]:
    articles = {}
    for row in chunks:
        article_no = str(row.get('article_number') or '').strip()
        if not article_no:
            continue
        articles[article_no] = {
            'article_number': article_no,
            'article_title': row.get('article_title', f'Dieu {article_no}'),
            'full_text': row.get('text', ''),
            'chunk_id': row.get('chunk_id', ''),
        }
    return articles


def shorten(text: str, max_len: int = 240) -> str:
    text = normalize_ws(text)
    if len(text) <= max_len:
        return text
    return text[: max_len - 3].rstrip() + '...'


def diff_excerpt(text: str, start: int, end: int, window: int = 120) -> str:
    left = max(0, start - window)
    right = min(len(text), end + window)
    return shorten(text[left:right].strip(), max_len=300)


def extract_evidence(before_text: str | None, after_text: str | None, max_items: int = 3) -> list[dict]:
    before_norm = normalize_ws(before_text or '')
    after_norm = normalize_ws(after_text or '')

    if before_norm and not after_norm:
        return [{'tag': 'removed', 'before': shorten(before_norm, 300), 'after': ''}]
    if after_norm and not before_norm:
        return [{'tag': 'added', 'before': '', 'after': shorten(after_norm, 300)}]

    matcher = difflib.SequenceMatcher(None, before_norm, after_norm)
    evidence = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            continue
        evidence.append({
            'tag': 'changed' if tag == 'replace' else tag,
            'before': diff_excerpt(before_norm, i1, i2),
            'after': diff_excerpt(after_norm, j1, j2),
        })
        if len(evidence) >= max_items:
            break
    return evidence


def parse_first_json_object(text: str) -> dict | None:
    if not text:
        return None
    raw = text.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    match = re.search(r'\{[\s\S]*\}', raw)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def _status_from_texts(before_norm: str, after_norm: str) -> str:
    if before_norm == after_norm:
        return 'unchanged'
    if before_norm and after_norm:
        return 'changed'
    if before_norm and not after_norm:
        return 'removed'
    return 'added'


def _normalize_evidence_items(evidence: list[dict], max_items: int = 3) -> list[dict]:
    normalized = []
    for item in evidence or []:
        if not isinstance(item, dict):
            continue
        before_part = shorten(str(item.get('before') or ''), 300)
        after_part = shorten(str(item.get('after') or ''), 300)
        if not before_part and not after_part:
            continue
        normalized.append({
            'tag': str(item.get('tag') or 'changed'),
            'before': before_part,
            'after': after_part,
        })
        if len(normalized) >= max_items:
            break
    return normalized


def enforce_no_evidence_no_conclusion(status: str, conclusion: str, evidence: list[dict], before_norm: str, after_norm: str):
    final_status = status if status in {'unchanged', 'changed', 'added', 'removed'} else _status_from_texts(before_norm, after_norm)
    final_evidence = _normalize_evidence_items(evidence)

    if final_status != 'unchanged' and not final_evidence:
        final_evidence = _normalize_evidence_items(extract_evidence(before_norm, after_norm, max_items=2), max_items=2)

    if final_status == 'unchanged':
        final_conclusion = 'Khong ghi nhan khac biet ve mat van ban.'
    elif final_evidence:
        cleaned = normalize_ws(conclusion or '')
        if cleaned:
            final_conclusion = cleaned
        elif final_status == 'added':
            final_conclusion = 'Dieu khoan duoc bo sung trong v2.'
        elif final_status == 'removed':
            final_conclusion = 'Dieu khoan bi loai bo so voi v1.'
        else:
            final_conclusion = 'Dieu khoan co thay doi noi dung giua v1 va v2.'
    else:
        final_conclusion = 'Khong du bang chung de ket luan; can kiem tra thu cong.'

    grounded = bool(final_evidence) or final_status == 'unchanged'
    return final_status, final_conclusion, final_evidence, grounded


def llm_compare_article(article_no: str, title: str, before_text: str | None, after_text: str | None, model: str = LLM_MODEL) -> dict:
    before_norm = normalize_ws(before_text or '')
    after_norm = normalize_ws(after_text or '')

    if before_norm == after_norm:
        return {
            'status': 'unchanged',
            'conclusion': 'Khong ghi nhan khac biet ve mat van ban.',
            'evidence': [],
            'llm_model': None,
            'llm_used': False,
            'fallback_reason': 'texts_equal',
            'grounded': True,
        }

    prompt = f"""
Ban la tro ly doi chieu hop dong. So sanh 2 PHIEN BAN cua CUNG MOT DIEU.

Muc tieu:
- Xac dinh trang thai thay doi.
- Dua ra ket luan ngan gon.
- Trich dan bang chung tu chinh van ban da cho.

QUY TAC BAT BUOC:
1) Chi su dung noi dung trong 'Dieu v1' va 'Dieu v2'. Khong bo sung kien thuc ngoai.
2) Trang thai CHI duoc la mot trong: unchanged, changed, added, removed.
3) Neu khong tim thay bang chung text ro rang, evidence phai de rong va conclusion ghi ro "Khong du bang chung de ket luan".
4) Tra ve DUY NHAT JSON dung schema, KHONG them markdown hay giai thich.

SCHEMA JSON:
{{
  "status": "unchanged|changed|added|removed",
  "conclusion": "string",
  "evidence": [
    {{"tag": "changed|added|removed", "before": "string", "after": "string"}}
  ]
}}

META:
- article_number: {article_no}
- article_title: {title}

Dieu v1:
{before_norm or '(khong co)'}

Dieu v2:
{after_norm or '(khong co)'}
""".strip()

    if ollama is None:
        status, conclusion, evidence, grounded = enforce_no_evidence_no_conclusion(
            _status_from_texts(before_norm, after_norm),
            'Khong goi duoc LLM (missing ollama package).',
            extract_evidence(before_norm, after_norm, max_items=2),
            before_norm,
            after_norm,
        )
        return {
            'status': status,
            'conclusion': conclusion,
            'evidence': evidence,
            'llm_model': None,
            'llm_used': False,
            'fallback_reason': 'missing_ollama_package',
            'grounded': grounded,
        }

    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0},
        )
        content = (response.get('message') or {}).get('content', '')
        parsed = parse_first_json_object(content)
        if not parsed:
            raise ValueError('LLM output is not valid JSON')

        status = parsed.get('status')
        conclusion = parsed.get('conclusion') or ''
        evidence = parsed.get('evidence') if isinstance(parsed.get('evidence'), list) else []

        status, conclusion, evidence, grounded = enforce_no_evidence_no_conclusion(
            status,
            conclusion,
            evidence,
            before_norm,
            after_norm,
        )

        return {
            'status': status,
            'conclusion': conclusion,
            'evidence': evidence,
            'llm_model': model,
            'llm_used': True,
            'fallback_reason': None,
            'grounded': grounded,
        }
    except Exception as exc:
        status, conclusion, evidence, grounded = enforce_no_evidence_no_conclusion(
            _status_from_texts(before_norm, after_norm),
            f'LLM fail ({type(exc).__name__}), fallback rule-based.',
            extract_evidence(before_norm, after_norm, max_items=2),
            before_norm,
            after_norm,
        )
        return {
            'status': status,
            'conclusion': conclusion,
            'evidence': evidence,
            'llm_model': model,
            'llm_used': False,
            'fallback_reason': str(exc),
            'grounded': grounded,
        }


def _distance_to_similarity(distance: float | int | None) -> float:
    if distance is None:
        return 0.0
    return 1.0 / (1.0 + float(distance))


def query_candidates_for_article(article_text: str, target_version: str, top_k: int = VECTOR_TOP_K) -> list[dict]:
    collection = get_collection(CHROMA_DIR, collection_name='legal_chunks')
    query_text = ViTokenizer.tokenize(normalize_ws(article_text))
    query_vector = embedder.encode([query_text], convert_to_numpy=True)[0].tolist()

    result = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        where={'version': target_version},
        include=['documents', 'metadatas', 'distances'],
    )

    candidates = []
    ids = (result.get('ids') or [[]])[0]
    documents = (result.get('documents') or [[]])[0]
    metadatas = (result.get('metadatas') or [[]])[0]
    distances = (result.get('distances') or [[]])[0]

    for idx, cand_id in enumerate(ids):
        md = metadatas[idx] if idx < len(metadatas) and isinstance(metadatas[idx], dict) else {}
        distance = distances[idx] if idx < len(distances) else None
        candidates.append({
            'chunk_id': cand_id,
            'article_number': str(md.get('article_number') or ''),
            'article_title': str(md.get('article_title') or ''),
            'text': documents[idx] if idx < len(documents) else '',
            'distance': distance,
            'similarity': _distance_to_similarity(distance),
            'metadata': md,
        })
    return candidates


def compare_articles_with_vector_retrieval(chunks_left: list[dict], chunks_right: list[dict]) -> list[dict]:
    left_articles = build_articles_from_chunks(chunks_left)
    right_articles = build_articles_from_chunks(chunks_right)

    left_numbers = sorted(left_articles.keys(), key=article_sort_key)
    matched_right = set()
    results = []

    for article_no in left_numbers:
        left = left_articles[article_no]
        candidates = query_candidates_for_article(left['full_text'], target_version='v2', top_k=VECTOR_TOP_K)

        chosen = None
        for cand in candidates:
            cand_article_no = cand.get('article_number')
            if not cand_article_no or cand_article_no in matched_right:
                continue
            if cand['similarity'] >= VECTOR_MATCH_THRESHOLD:
                chosen = cand
                break

        if chosen is None:
            llm_result = llm_compare_article(
                article_no=article_no,
                title=left['article_title'],
                before_text=left['full_text'],
                after_text=None,
            )
            results.append({
                'article_number': article_no,
                'article_title': left['article_title'],
                'matched_article_v2': None,
                'match_score': 0.0,
                'status': llm_result['status'],
                'conclusion': llm_result['conclusion'],
                'evidence': llm_result['evidence'],
                'grounded': llm_result['grounded'],
                'llm_model': llm_result['llm_model'],
                'llm_used': llm_result['llm_used'],
                'fallback_reason': llm_result['fallback_reason'],
                'v1_text': left['full_text'],
                'v2_text': None,
            })
            continue

        matched_article_no = chosen['article_number']
        matched_right.add(matched_article_no)
        right = right_articles.get(matched_article_no)
        right_text = right['full_text'] if right else chosen['text']
        title = (right or {}).get('article_title', chosen.get('article_title') or left['article_title'])

        llm_result = llm_compare_article(
            article_no=article_no,
            title=title,
            before_text=left['full_text'],
            after_text=right_text,
        )

        results.append({
            'article_number': article_no,
            'article_title': title,
            'matched_article_v2': matched_article_no,
            'match_score': round(float(chosen['similarity']), 4),
            'status': llm_result['status'],
            'conclusion': llm_result['conclusion'],
            'evidence': llm_result['evidence'],
            'grounded': llm_result['grounded'],
            'llm_model': llm_result['llm_model'],
            'llm_used': llm_result['llm_used'],
            'fallback_reason': llm_result['fallback_reason'],
            'v1_text': left['full_text'],
            'v2_text': right_text,
        })

    for article_no in sorted(right_articles.keys(), key=article_sort_key):
        if article_no in matched_right:
            continue
        right = right_articles[article_no]
        llm_result = llm_compare_article(
            article_no=article_no,
            title=right['article_title'],
            before_text=None,
            after_text=right['full_text'],
        )
        results.append({
            'article_number': article_no,
            'article_title': right['article_title'],
            'matched_article_v2': article_no,
            'match_score': 0.0,
            'status': llm_result['status'],
            'conclusion': llm_result['conclusion'],
            'evidence': llm_result['evidence'],
            'grounded': llm_result['grounded'],
            'llm_model': llm_result['llm_model'],
            'llm_used': llm_result['llm_used'],
            'fallback_reason': llm_result['fallback_reason'],
            'v1_text': None,
            'v2_text': right['full_text'],
        })

    return results


comparison_results = compare_articles_with_vector_retrieval(chunks_v1, chunks_v2)
status_counts = Counter(item['status'] for item in comparison_results)
grounded_count = sum(1 for item in comparison_results if item['grounded'])
ungrounded_count = len(comparison_results) - grounded_count

{
    'total_articles': len(comparison_results),
    'status_counts': status_counts,
    'grounded_count': grounded_count,
    'ungrounded_count': ungrounded_count,
    'vector_top_k': VECTOR_TOP_K,
    'vector_match_threshold': VECTOR_MATCH_THRESHOLD,
}

{'total_articles': 21,
 'status_counts': Counter({'unchanged': 16, 'changed': 5}),
 'grounded_count': 21,
 'ungrounded_count': 0,
 'vector_top_k': 3,
 'vector_match_threshold': 0.5}

In [6]:
# === Cell 6: Bao cao ket qua so sanh + citation ===

print('=== TONG QUAN SO SANH O MUC DIEU (LLM + VECTOR RETRIEVAL) ===')
print(f"Tong dieu doi chieu : {len(comparison_results)}")
for status in ['unchanged', 'changed', 'added', 'removed']:
    print(f"- {status:<9}: {status_counts.get(status, 0)}")

llm_used_count = sum(1 for item in comparison_results if item['llm_used'])
fallback_count = len(comparison_results) - llm_used_count
print(f"- LLM da xu ly      : {llm_used_count}")
print(f"- Fallback rule-based: {fallback_count}")
print(f"- Co bang chung (grounded): {grounded_count}")
print(f"- Khong du bang chung     : {ungrounded_count}")
print(f"- Vector top_k            : {VECTOR_TOP_K}")
print(f"- Match threshold         : {VECTOR_MATCH_THRESHOLD}")

summary_df = pd.DataFrame([
    {
        'Dieu v1': item['article_number'],
        'Dieu v2 match': item['matched_article_v2'] or '(khong tim thay)',
        'Match score': item['match_score'],
        'Tieu de': item['article_title'],
        'Trang thai': item['status'],
        'Grounded': item['grounded'],
        'Ket luan': shorten(item['conclusion'], 120),
    }
    for item in comparison_results
])

changed_df = summary_df[summary_df['Trang thai'] != 'unchanged'].copy()
print('\n=== CAC DIEU CO THAY DOI / THEM / BO ===')
display(changed_df if not changed_df.empty else pd.DataFrame(columns=['Dieu v1', 'Dieu v2 match', 'Match score', 'Tieu de', 'Trang thai', 'Grounded', 'Ket luan']))

citation_rows = []
for item in comparison_results:
    if item['status'] == 'unchanged':
        continue
    if not item['evidence']:
        citation_rows.append({
            'Dieu v1': item['article_number'],
            'Dieu v2 match': item['matched_article_v2'] or '(khong tim thay)',
            'Loai': 'No evidence',
            'V1': '',
            'V2': '',
        })
        continue
    for ev in item['evidence']:
        citation_rows.append({
            'Dieu v1': item['article_number'],
            'Dieu v2 match': item['matched_article_v2'] or '(khong tim thay)',
            'Loai': ev.get('tag', 'changed'),
            'V1': ev.get('before', ''),
            'V2': ev.get('after', ''),
        })

citation_df = pd.DataFrame(citation_rows)
print('\n=== BANG TRICH DAN (CITATION) ===')
display(citation_df if not citation_df.empty else pd.DataFrame(columns=['Dieu v1', 'Dieu v2 match', 'Loai', 'V1', 'V2']))

report_payload = {
    'config': {
        'file_v1': FILE_V1.name,
        'file_v2': FILE_V2.name,
        'llm_model': LLM_MODEL,
        'principle': 'Khong bang chung -> khong ket luan',
        'total_articles': len(comparison_results),
        'llm_used_count': llm_used_count,
        'fallback_count': fallback_count,
        'grounded_count': grounded_count,
        'ungrounded_count': ungrounded_count,
        'vector_top_k': VECTOR_TOP_K,
        'vector_match_threshold': VECTOR_MATCH_THRESHOLD,
    },
    'article_level_results': comparison_results,
}

ARTICLE_REPORT_JSON = OUTPUT_DIR / 'sample_pair_article_llm_vector_report.json'
ARTICLE_REPORT_JSON.write_text(json.dumps(report_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('\n[OK] Da luu bao cao JSON:', ARTICLE_REPORT_JSON)

print('\n=== NGUYEN TAC BAO CAO ===')
print('- Cap doi v1-v2 duoc ghep bang vector retrieval.')
print('- Chi ket luan tren thay doi khi co evidence hop le.')
print('- Neu khong du bang chung, he thong tra ve: can kiem tra thu cong.')

=== TONG QUAN SO SANH O MUC DIEU (LLM + VECTOR RETRIEVAL) ===
Tong dieu doi chieu : 21
- unchanged: 16
- changed  : 5
- added    : 0
- removed  : 0
- LLM da xu ly      : 5
- Fallback rule-based: 16
- Co bang chung (grounded): 21
- Khong du bang chung     : 0
- Vector top_k            : 3
- Match threshold         : 0.5

=== CAC DIEU CO THAY DOI / THEM / BO ===


,Dieu v1,Dieu v2 match,Match score,Tieu de,Trang thai,Grounded,Ket luan
0,1,1,0.9688,Đối Tượng Của Hợp Đồng,changed,True,Diện tích đất của Khu đất đã được thay đổi từ ...
3,4,4,0.5030,Vốn Điều Lệ,changed,True,"Vốn điều lệ và số lượng cổ phần đã thay đổi, c..."
5,6,6,0.8778,Thời Hạn Hoạt Động Và Chấm Dứt Hợp Đồng,changed,True,Thời hạn hoạt động của công ty được thay đổi t...
10,11,11,0.9912,Tổ Chức Và Quản Lý Công Ty,changed,True,Số lần họp Đại hội đồng cổ đông được tăng từ m...
20,21,21,0.8477,Thỏa Thuận Khác,changed,True,Ngày ký hợp đồng và số bản chính đã thay đổi.



=== BANG TRICH DAN (CITATION) ===


,Dieu v1,Dieu v2 match,Loai,V1,V2
0,1,1,changed,12.500 m²,15.500 m²
1,4,4,changed,100.000.000.000 VNĐ (Một trăm tỷ đồng Việt Nam),130.000.000.000 VNĐ (Một trăm ba mươi tỷ đồng ...
2,4,4,changed,10.000.000 cổ phần,13.000.000 cổ phần
3,4,4,changed,Bên A sẽ góp vào Vốn Điều Lệ 40.000.000.000 VNĐ,Bên A sẽ góp vào Vốn Điều Lệ 55.000.000.000 VNĐ
4,6,6,changed,Thời Hạn Hoạt Động của Công ty sẽ là 50 năm,Thời Hạn Hoạt Động của Công ty sẽ là 55 năm
5,11,11,changed,Đại hội đồng Cổ đông họp ít nhất mỗi năm một lần.,Đại hội đồng Cổ đông họp ít nhất mỗi năm hai lần.
6,21,21,changed,15 tháng 04 năm 2026,20 tháng 04 năm 2026
7,21,21,changed,06 (sáu) bản chính,08 (tám) bản chính



[OK] Da luu bao cao JSON: d:\Data Self Learning\Extra Projects\rag-based-legal-comparison\outputs\sample_pair_article_llm_vector_report.json

=== NGUYEN TAC BAO CAO ===
- Cap doi v1-v2 duoc ghep bang vector retrieval.
- Chi ket luan tren thay doi khi co evidence hop le.
- Neu khong du bang chung, he thong tra ve: can kiem tra thu cong.
